In [1]:
from Montreal_UHI_toolbox import stations
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
import xarray as xr
from skimage.transform import resize

In [2]:
data = stations.sel(time=slice('1980','2022'))
station_names = data.station_name.values
times = data.time.values

In [ ]:
# Creating 3 distinct graphs to display the temperature data
for field,title in zip(['tasmin','tasmax','tas'],
                       ['Minimum','Maximum','Average']):
    hover_text = np.where(np.isnan(data[field].values),f'[missing {field} data]', data[field].values.astype(object))
    fig = go.Figure(go.Heatmap(
        z = data[field],
        colorscale=px.colors.sequential.RdBu_r,
        x = times,
        y = station_names,
        zmin = -40,
        zmax = 40,
        text=hover_text,
        hovertemplate=(
            'date: %{x}<br>' + 
            'station: %{y}<br>' +
            '%{text}°C<extra></extra>'
    
        ),
        colorbar=dict(title='Temperature (°C)') ))
    
    fig.update_layout(
        xaxis_title='Time', 
        yaxis_title='Station',
        template='simple_white',
        height=900,
        title= f'1980-2022 ECCC Gen3 Homogenised {title} Daily Temperature Records for Select Stations'
    )
    
    fig.show()

In [ ]:
# It may be easier to visualise in terms of data availability
# for each moment in time, mask[field] is 1 if data exists, 0 if data does not exist
mask = {}
for field in ['tasmin','tasmax','tas']:
    mask[field] = (~np.isnan(data[field])).astype(int)
    mask[field].attrs['units'] = 'binary'
    mask[field].attrs['long_name'] = 'Data Presence'

# Example with tasmin
field = 'tasmin'
hover_text = np.where(mask[field] == 1, f'{field} data exists: yes', f'{field} data exists: no')
fig = go.Figure(go.Heatmap(
    z=mask[field],
    x=times,
    y=station_names,
    colorscale=[[0, 'white'], [0.5, 'white'], [0.5, 'black'], [1, 'black']], 
    zmin=0,
    zmax=1,
    colorbar=dict(
        tickvals=[0.25, 0.75],
        ticktext=['No','Yes'],
        title='Data Exists'
    ),
    text=hover_text,
    hovertemplate=(
        'time: %{x}<br>' + 
        'station: %{y}<br>' +
        '%{text}<extra></extra>' 
    )
))

fig.update_layout(
    xaxis_title='Time', 
    yaxis_title='Station',
    height=900,
    template='simple_white'
)

fig.show()

In [12]:
# Ultimately all three may be plotted on the same graph with toggles between them
traces = []
for field, title in zip(['tasmin','tasmax','tas'],
                        ['Minimum','Maximum','Average']):
    hover_text = np.where(np.isnan(data[field].values),
                          f'[missing {field} data]',
                          data[field].values.astype(object))
    
    trace = go.Heatmap(
        z=data[field],
        colorscale=px.colors.sequential.RdBu_r,
        x=times,
        y=station_names,
        zmin=-40,
        zmax=40,
        text=hover_text,
        hovertemplate=(
            'date: %{x}<br>' + 
            'station: %{y}<br>' +
            '%{text}°C<extra></extra>'
        ),
        colorbar=dict(title='Temperature (°C)'),
        visible=True if field=='tasmin' else False  # Show tasmin first by default
    )
    traces.append(trace)

# Figure with all traces
fig = go.Figure(data=traces)

# Radio buttons for toggling between tasmin, tasmax, and tas
fig.update_layout(
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=1.0,
            y=1.05,
            showactive=True,
            buttons=list([
                dict(label='Min',
                     method='update',
                     args=[{'visible': [True, False, False]},
                           {'title': {'text':'1980-2022 ECCC Gen3 Homogenised Minimum Daily Temperature Records for Select Stations'}}]),
                dict(label='Max',
                     method='update',
                     args=[{'visible': [False, True, False]},
                           {'title': {'text': '1980-2022 ECCC Gen3 Homogenised Maximum Daily Temperature Records for Select Stations'}}]),
                dict(label='Avg',
                     method='update',
                     args=[{'visible': [False, False, True]},
                           {'title': {'text': '1980-2022 ECCC Gen3 Homogenised Average Daily Temperature Records for Select Stations'}}]),
            ]),
        )
    ],
    xaxis_title='Time', 
    yaxis_title='Station',
    template='simple_white',
    title='1980-2022 ECCC Gen3 Minimum Homogenised Daily Temperature Records for Select Stations', # (default minimum)
    height=900
)

# fig.show()
fig.write_html('/home/gulley/UHI_HW_MTL/info/plots/station_daily_data_1980_2022.html')